# Stage 34A: OOF-selected conservative TCN gate

Stage 33Aの保存済みreportだけを使い、training OOFからhard-gate閾値を固定します。CPU専用で、再学習・TCN再推論・予約120 wellsの使用・提出作成はありません。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess

REPOSITORY_URL = 'https://github.com/Okada-N13/rogii.git'
repo_dir = Path('/content/ROGII')
artifact_dir = Path('/content/drive/MyDrive/kaggle/rogii/artifacts')

if not (repo_dir / '.git').is_dir():
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(repo_dir)], check=True)
else:
    subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only', 'origin', 'main'], check=True)
if shutil.which('uv') is None:
    subprocess.run(['bash', '-lc', 'curl -LsSf https://astral.sh/uv/install.sh | sh'], check=True)
os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']
subprocess.run(['uv', 'sync', '--frozen'], cwd=repo_dir, check=True)
print('Stage 34A is CPU-only; GPU runtime is unnecessary.')


In [ ]:
stage33a_run = artifact_dir / 'stage33a_tcn_cut_benefit_gate_full_v001'
required = [
    stage33a_run / 'summary.json',
    stage33a_run / 'training_gate_report.parquet',
    stage33a_run / 'validation_gate_report.parquet',
]
for path in required:
    assert path.is_file(), path
stage33 = json.loads((stage33a_run / 'summary.json').read_text())
assert stage33['stage33a_complete'] is True
assert stage33['reserved_confirmation_used'] is False
print(stage33['training_cuts'], 'OOF cuts ready; reserve remains sealed')


In [ ]:
RUN_ID = 'stage34a_selective_tcn_gate_full_v001'
run_dir = artifact_dir / RUN_ID
if not (run_dir / 'summary.json').is_file():
    result = subprocess.run([
        'uv', 'run', 'rogii-residual-selective-gate',
        '--config', 'configs/experiment/stage34a_selective_tcn_gate.yaml',
        '--stage33a-run', str(stage33a_run),
        '--artifact-dir', str(artifact_dir),
        '--run-id', RUN_ID,
    ], cwd=repo_dir, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode:
        raise RuntimeError(f'Stage 34A failed with exit code {result.returncode}')
else:
    print('Reusing completed run:', run_dir)


In [ ]:
import pandas as pd

summary = json.loads((run_dir / 'summary.json').read_text())
display(pd.DataFrame(summary['threshold_audits']).sort_values('threshold'))
display(pd.DataFrame(summary['family_reports']['stage16_fold']['fold_report']))
{key: summary[key] for key in [
    'stage34a_complete',
    'promoted_to_stage34b_reserved_confirmation',
    'selection_role',
    'threshold_candidates',
    'eligible_thresholds',
    'selected_threshold',
    'validation_active_fraction',
    'validation_active_cuts',
    'base_rmse',
    'candidate_rmse',
    'rmse_delta',
    'bootstrap_95pct',
    'cut_rmse_p90_delta',
    'gates',
    'reserved_confirmation_used',
    'next_step',
]}
